# SAP HANA Cloud Checkpointer for LangGraph

[LangGraph](https://langchain-ai.github.io/langgraph/) persists the state of a run in a **checkpointer** so that agents and conversations are durable: a thread can be paused and resumed, inspected, rewound ("time travel"), and shared across processes.

`HanaCheckpointSaver` (and its native-async twin `AsyncHanaCheckpointSaver`) store those checkpoints and the pending writes of each step in **SAP HANA Cloud**, so your LangGraph application state lives in the same database as the rest of your data.

What gets persisted per thread:
- **checkpoints** — a snapshot of the graph channels after each step,
- **blobs** — the (large) channel values, stored once and referenced by version,
- **pending writes** — writes emitted by a step before the next checkpoint, so an interrupted run resumes correctly.

## Setup & installation

Install `langchain-hana`. LangGraph is pulled in automatically as a dependency, so no
separate install is needed:

In [ ]:
pip install langchain-hana

First, open a connection to your SAP HANA Cloud instance with `hdbcli`.

In [2]:
import os

from dotenv import load_dotenv
from hdbcli import dbapi

# Load HANA_DB_ADDRESS / HANA_DB_PORT / HANA_DB_USER / HANA_DB_PASSWORD from a .env
load_dotenv()

connection = dbapi.connect(
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    user=os.environ.get("HANA_DB_USER"),
    password=os.environ.get("HANA_DB_PASSWORD"),
)

## Creating a checkpointer

Pass the connection to `HanaCheckpointSaver`. The constructor is self-initializing: it applies the (idempotent) schema migrations, creating its tables on first use. It also switches the connection to manual-commit mode so that each multi-statement checkpoint write commits atomically.

> **Note:** the migrations are versioned, so constructing the saver again against the same schema is cheap and safe — only new migrations are applied.

In [3]:
from langchain_hana import HanaCheckpointSaver

checkpointer = HanaCheckpointSaver(connection)

/tmp/ipykernel_92539/1778353732.py:3: UserWarning: HanaCheckpointSaver is disabling autocommit on the provided connection so checkpoint writes commit atomically.
  checkpointer = HanaCheckpointSaver(connection)


### Constructor options

By default the saver creates its tables in the connection's current schema with standard names. You can override the schema and table names — useful to isolate environments or run several independent applications side by side.

In [ ]:
checkpointer = HanaCheckpointSaver(
    connection,
    schema_name="MY_APP",  # defaults to the connection's current schema
    migrations_table="LG_CKPT_MIGRATIONS",
    checkpoints_table="LG_CHECKPOINTS",
    blobs_table="LG_CHECKPOINT_BLOBS",
    writes_table="LG_CHECKPOINT_WRITES",
)

## Using the checkpointer with an agent

Pass the checkpointer to a LangChain agent via `checkpointer=...`. The checkpointer makes the agent's conversation durable: every invocation that carries a `thread_id` is persisted, so follow-up turns on the same thread see the earlier messages — the agent's short-term (within-thread) memory.

> The agent uses LangChain's `create_agent` with an LLM from the SAP generative AI hub SDK (`pip install sap-ai-sdk-gen`); any LangChain chat model works.

In [4]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from langchain.agents import create_agent

# LLM from the SAP generative AI hub SDK (sap-ai-sdk-gen)
llm = ChatOpenAI(proxy_model_name="gpt-4o", temperature=0)
# Or any LangChain chat model, e.g.:
#   from langchain_openai import ChatOpenAI; llm = ChatOpenAI(model="gpt-4o")


def get_weather(city: str) -> str:
    """Return the weather for a city."""
    return f"It's always sunny in {city}."


agent = create_agent(
    model=llm,
    tools=[get_weather],
    checkpointer=checkpointer,
)

Run the agent on a thread. The `thread_id` is the durable key under which all checkpoints for this conversation are stored.

In [5]:
thread = {"configurable": {"thread_id": "demo-thread"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, I'm Alice. Weather in Paris?"}]},
    thread,
)

{'messages': [HumanMessage(content="Hi, I'm Alice. Weather in Paris?", additional_kwargs={}, response_metadata={}, id='7114c81f-13d7-431c-bc72-04730ace97d7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 53, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 27, 'engine_ttlt_ms': 114, 'pre_inference_ms': 106, 'service_tbt_ms': 5, 'service_ttft_ms': 211, 'service_ttlt_ms': 288, 'user_visible_ttft_ms': 105}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9f2f71270d', 'id': 'chatcmpl-EmJ86i9t5TvM0x7Q2NfCjd5kNY7x7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc

### State persists across runs

Because the conversation is stored in HANA, a follow-up on the **same** thread resumes from the saved messages — even from a brand-new process or after a restart. The agent still knows the earlier turn.

In [6]:
followup = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    thread,
)
print(followup["messages"][-1].content)  # answers "Alice" from the persisted history

Your name is Alice.


### Inspecting state and history

The agent is a compiled graph, so the standard introspection API works. `get_state` returns the latest checkpoint for a thread; `get_state_history` walks every checkpoint (newest first), which is what powers time-travel and debugging.

In [7]:
state = agent.get_state(thread)
print("messages so far:", len(state.values["messages"]))

for snapshot in agent.get_state_history(thread):
    ckpt_id = snapshot.config["configurable"]["checkpoint_id"]
    print(ckpt_id, "->", len(snapshot.values.get("messages", [])), "messages")

messages so far: 6


1f1e4e7d-ae58-672b-8006-e0bfa9d3225a -> 6 messages
1f1e4e7d-a832-6be3-8005-b2ad904163db -> 5 messages
1f1e4e7d-a830-602a-8004-ca051cfd8951 -> 4 messages
1f1e4e7d-a6d1-62b8-8003-188eeebec856 -> 4 messages
1f1e4e7d-9e69-6685-8002-b5c9dbfa5f19 -> 3 messages
1f1e4e7d-9e63-6506-8001-33b2d108f114 -> 2 messages
1f1e4e7d-9699-6847-8000-cf06916cd06e -> 1 messages
1f1e4e7d-9696-62fc-bfff-9d714cc8173b -> 0 messages


### Listing checkpoints directly

The saver also exposes the low-level checkpointer API. `list()` streams checkpoint tuples for a thread, with optional `limit`, metadata `filter`, and a `before` cursor.

In [8]:
for ckpt in checkpointer.list(thread, limit=5):
    print(ckpt.config["configurable"]["checkpoint_id"], ckpt.metadata.get("source"))

1f1e4e7d-ae58-672b-8006-e0bfa9d3225a loop
1f1e4e7d-a832-6be3-8005-b2ad904163db loop
1f1e4e7d-a830-602a-8004-ca051cfd8951 input
1f1e4e7d-a6d1-62b8-8003-188eeebec856 loop
1f1e4e7d-9e69-6685-8002-b5c9dbfa5f19 loop


## Asynchronous usage

`AsyncHanaCheckpointSaver` drives an `hdbcli` **async** connection. Two differences from the sync saver:

1. open the connection with `await dbapi.async_connect(...)`, and
2. call `await checkpointer.setup()` once before use — the async saver does **not** set up in its constructor (it cannot `await` there).

Build the agent with the async saver and use the async API (`ainvoke`, ...).

In [9]:
from langchain_hana import AsyncHanaCheckpointSaver

async_connection = await dbapi.async_connect(
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    user=os.environ.get("HANA_DB_USER"),
    password=os.environ.get("HANA_DB_PASSWORD"),
)

checkpointer = AsyncHanaCheckpointSaver(async_connection)
await checkpointer.setup()  # required once; creates/updates the tables

agent = create_agent(model=llm, tools=[get_weather], checkpointer=checkpointer)

thread = {"configurable": {"thread_id": "demo-thread-async"}}
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Weather in Berlin?"}]},
    thread,
)
print(result["messages"][-1].content)

/tmp/ipykernel_92539/848711103.py:10: UserWarning: AsyncHanaCheckpointSaver is disabling autocommit on the provided connection so checkpoint writes commit atomically.
  checkpointer = AsyncHanaCheckpointSaver(async_connection)


The weather in Berlin is currently sunny.


## Cleanup

Close the connection(s) when done.

In [10]:
connection.close()
await async_connection.close()